In [1]:
"""Show what's stored in the agent memory SQLite file.   python inspect_memory.py agent_memory.db"""
import json
import sqlite3
import sys
import pandas as pd


# Ignore Jupyter's kernel arguments.
if len(sys.argv) > 1 and sys.argv[1].startswith("--f="):
    sys.argv = sys.argv[:1]

path = sys.argv[1] if len(sys.argv) > 1 else "agent_memory.db"
db = sqlite3.connect(f"file:{path}?mode=ro", uri=True)

tables = [r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print("collections:", tables)

for t in tables:
    rows = db.execute(f'SELECT key, data FROM "{t}"').fetchall()
    print(f"\n=== {t}: {len(rows)} record(s) ===")
    for key, data in rows:
        d = json.loads(data)
        emb = d.pop("text_embedding", None)
        d["text_embedding"] = f"<{len(emb)} floats>" if emb else None
        for k in ("definition", "description", "sample_data", "instructions"):
            if d.get(k):
                d[k] = d[k][:100].replace("\n", " ") + ("..." if len(d[k]) > 100 else "")
        print(key, json.dumps(d, indent=2))

collections: ['tables', 'agent']

=== tables: 13 record(s) ===
6f7cc2b0-15d5-422a-a453-404234b5278e {
  "table_name": "[Categories]",
  "key": "6f7cc2b0-15d5-422a-a453-404234b5278e",
  "definition": "| cid | name | type | notnull | dflt_value | pk | | --- | --- | --- | --- | --- | --- | | 0 | Catego...",
  "description": "### [Categories]  This table is used to store information about different product categories, includ...",
  "sample_data": "| CategoryID | CategoryName | Description | Picture | | --- | --- | --- | --- | | 1 | Beverages | So...",
  "text_embedding": "<256 floats>"
}
e878e688-c090-4eba-9903-537195c4b9ac {
  "table_name": "[CustomerCustomerDemo]",
  "key": "e878e688-c090-4eba-9903-537195c4b9ac",
  "definition": "| cid | name | type | notnull | dflt_value | pk | | --- | --- | --- | --- | --- | --- | | 0 | Custom...",
  "description": "### [CustomerCustomerDemo]  This table appears to be a junction table that links customers to their ...",
  "sample_data": "| CustomerID |

In [2]:
pd.set_option("display.max_colwidth", None)

agents_df = pd.DataFrame([json.loads(d) for (d,) in db.execute('SELECT data FROM "agent"')])
agents_df["embedding_size"] = agents_df["text_embedding"].apply(lambda e: len(e) if e else 0)
agents_df = agents_df.drop(columns=["text_embedding"])
display(agents_df)

for _, a in agents_df.iterrows():
    print(f"\n=== {a['agent_name']} ===")
    print("DESCRIPTION:\n", a["description"])
    print("\nINSTRUCTIONS:\n", a["instructions"])

,agent_name,key,description,instructions,embedding_size
0,BizManger,00000000-0000-0000-0000-000000000000,"The agent is designed to manage and facilitate the operations related to product categories, customer demographics, and employee territories within a retail or business management system. It ensures the accurate and efficient handling of customer and product data, supporting various business processes such as sales, inventory management, and customer relationship management.\n\n### Relationships\n- **Categories**: This table serves as a foundational reference for organizing and categorizing products. It is indirectly related to the [Products] table, where each product is linked to a specific category.\n- **CustomerCustomerDemo**: This table acts as a junction table, linking customers to their corresponding customer types. It establishes a many-to-many relationship between the [Customers] and [CustomerDemographics] tables.\n- **CustomerDemographics**: This table categorizes customers based on demographic types, linking to the [Customers] table through [CustomerTypeID].\n- **Customers**: This table stores detailed information about customer companies, including their contact details and addresses. It is related to the [Orders] table, where each order is associated with a specific customer.\n- **EmployeeTerritories**: This table establishes the relationship between employees and their assigned territories, linking to the [Employees] and [Territories] tables.\n- **Employees**: This table stores detailed information about employees, including their names, titles, and contact information. It is related to the [Orders] table, where each order is handled by one employee, and the [EmployeeTerritories] table, where each employee can be associated with multiple territories.\n- **Order Details**: This table stores detailed information about the products included in each order, linking to the [Orders] and [Products] tables.\n- **Orders**: This table manages and tracks orders, linking to the [Customers], [Employees], and [Shippers] tables.\n- **Products**: This table stores detailed information about each product, linking to the [Suppliers] and [Categories] tables.\n- **Regions**: This table stores information about different regions, linking to the [Territories] table.\n- **Shippers**: This table stores information about shipping companies, linking to the [Orders] table.\n- **Suppliers**: This table stores information about suppliers, linking to the [Products] table.\n- **Territories**: This table stores detailed information about specific territories within a region, linking to the [Regions] table.\n\n### Examples of actions supported\n- Search for orders, update product inventory, retrieve and manage customer details, assign territories to employees, and categorize products by their respective categories.","You are an agent designed to manage and facilitate operations related to product categories, customer demographics, and employee territories within a retail or business management system. Your primary objectives are to ensure the accurate and efficient handling of customer and product data, supporting various business processes such as sales, inventory management, and customer relationship management. You will use a set of tools to interact with multiple tables in a database, including Categories, CustomerCustomerDemo, CustomerDemographics, Customers, EmployeeTerritories, Employees, Order Details, Orders, Products, Regions, Shippers, Suppliers, and Territories. Here are the detailed instructions for your operation.",256



=== BizManger ===
DESCRIPTION:
 The agent is designed to manage and facilitate the operations related to product categories, customer demographics, and employee territories within a retail or business management system. It ensures the accurate and efficient handling of customer and product data, supporting various business processes such as sales, inventory management, and customer relationship management.

### Relationships
- **Categories**: This table serves as a foundational reference for organizing and categorizing products. It is indirectly related to the [Products] table, where each product is linked to a specific category.
- **CustomerCustomerDemo**: This table acts as a junction table, linking customers to their corresponding customer types. It establishes a many-to-many relationship between the [Customers] and [CustomerDemographics] tables.
- **CustomerDemographics**: This table categorizes customers based on demographic types, linking to the [Customers] table through [Custom

In [3]:
tables_df = pd.DataFrame([json.loads(d) for (d,) in db.execute('SELECT data FROM "tables"')])
tables_df["embedding_size"] = tables_df["text_embedding"].apply(lambda e: len(e) if e else 0)
tables_df = tables_df.drop(columns=["text_embedding"]).sort_values("table_name").reset_index(drop=True)

print(f"{len(tables_df)} table(s) memorized")
display(tables_df[["table_name", "embedding_size"]])

def show(name):
    t = tables_df[tables_df["table_name"] == name].iloc[0]
    print("DESCRIPTION:\n", t["description"])
    print("\nDEFINITION:\n", t["definition"])
    print("\nSAMPLE DATA:\n", t["sample_data"])

show(tables_df["table_name"].iloc[0])  # or show("[Customers]")

13 table(s) memorized


,table_name,embedding_size
0,[Categories],256
1,[CustomerCustomerDemo],256
2,[CustomerDemographics],256
3,[Customers],256
4,[EmployeeTerritories],256
5,[Employees],256
6,[Order Details],256
7,[Orders],256
8,[Products],256
9,[Regions],256


DESCRIPTION:
 ### [Categories]

This table is used to store information about different product categories, including their names, descriptions, and associated images. It serves as a foundational reference for organizing and categorizing products in a retail or inventory management system.

#### Attributes

- **CategoryID** (primary key): Unique identifier for each category.
- **CategoryName**: Name of the category.
- **Description**: Detailed description of the category.
- **Picture**: Binary data representing an image associated with the category.


#### Relations

This table does not explicitly define relationships with other tables based on the provided information. However, it could be related to a products table where each product is linked to a specific category.


DEFINITION:
 | cid | name | type | notnull | dflt_value | pk |
| --- | --- | --- | --- | --- | --- |
| 0 | CategoryID | INTEGER | 0 | None | 1 |
| 1 | CategoryName | TEXT | 0 | None | 0 |
| 2 | Description | TEXT | 0 